# “Style Translations”

<img src="https://live.staticflickr.com/65535/54530431746_73942f964a_n.jpg" alt="Embedded Photo" width="500">

*Image generated using the DALL·E model.*

## Introduction

Aiga went to a bookstore and bought a book about deep learning. She was very excited — finally she would properly refresh her knowledge before the finals of the Artificial Intelligence Olympiad!

She returned home, sat down comfortably with a cup of tea, and began reading the first sentences:

> *“Sentence embeddings obtained from encoder models are usually better than those from decoder-only models.”*  
>
> *“Reinforcement learning from human feedback is typically performed by optimizing a policy proximal to the reward model.”*

— What...? — Aiga muttered, looking confused at the pages.

She called her friend, who immediately explained:

> *“Sentence embeddings are better if you extract them from encoders rather than decoder-only models.”*  
>
> *“RLHF typically uses PPO to optimize the policy with respect to the reward model.”*

Aiga sighed with relief. Now everything became *a little* clearer.

But the relief was only temporary — she still could not accept the translation style in the book. So she decided to train her own English-to-Polish translation model — one that would translate AI-related terminology in a way she prefers!

## Task

Your goal in this task is to adapt translations generated by an existing machine translation model to a specific style — preserving the original wording of selected machine learning terms.

The base model used here is [MarianMT](https://huggingface.co/docs/transformers/en/model_doc/marian) — a model translating from English to Polish based on an encoder-decoder architecture. By default, this model translates all words, including specialized terminology. Your task is to implement a preprocessing function and fine-tune this model so that specialized terminology is not translated into Polish.

## Data

The available data consists of:

* **Training set** — contains English sentences, their Polish translations, and a list of keywords that should remain untranslated.

* **Validation set** — used to evaluate your approach during training; it has the same format as the training set.

Each example has the following format:

```json
{
    "keywords": ["explainable AI"],
    "en": "Developing explainable AI tools is crucial for trust in automated systems.",
    "pl": "Rozwijanie narzędzi explainable AI ma kluczowe znaczenie dla zaufania do zautomatyzowanych systemów."
}
```

The test set, which will be used for final evaluation, **will not contain Polish translations of sentences**.

## Evaluation Metric

Your solution will be evaluated on a hidden test set using the **BLEU** metric. The test set is similar to the validation set.

BLEU measures the overlap of *n-grams* (continuous sequences of *n* consecutive words) between your translation and a reference sentence — the higher the overlap, the better the score.

## Constraints

* Your solution will be tested on the competition platform without internet access and with **GPU support**.
* Training and evaluation must not exceed **10 minutes**.
* Allowed libraries: `torch`, `pandas`, `numpy`, `nltk`, `transformers`, `datasets`, `matplotlib`.

## Submission Files

* This notebook filled with your solution.

## Evaluation

Remember that during testing the flag `FINAL_EVALUATION_MODE` will be set to `True`.

You can score between 0 and 100 points for this task. If your model achieves a BLEU score lower than **0.82** on the hidden test set, you will receive 0 points. If it exceeds **0.86**, you will receive the maximum score. Between these thresholds, the score scales linearly and is rounded to an integer.

$$
\mathrm{points} =
\begin{cases}
100 & \text{if } \mathrm{BLEU} > 0.86 \
0 & \text{if } \mathrm{BLEU} < 0.82 \
100 \cdot \dfrac{\mathrm{BLEU} - 0.82}{0.86 - 0.82} & \text{otherwise}
\end{cases}
$$

## Starter Code

In this section, we initialize the environment by importing the necessary libraries and functions. The provided code will help you efficiently handle the data and build an appropriate solution.


In [2]:
######################### DO NOT CHANGE THIS CELL ##########################

FINAL_EVALUATION_MODE = True

import json
import os
import torch

import matplotlib.pyplot as plt
import numpy as np
from datasets import Dataset
from nltk.translate.bleu_score import sentence_bleu
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [3]:
######################### DO NOT CHANGE THIS CELL ##########################

from transformers import set_seed
import random

seed = 42

os.environ["PYTHONHASHSEED"] = str(seed)

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

set_seed(seed)

## Loading Data

In this part of the task, we will load the training data.

In [5]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
device

device(type='cuda')

In [6]:
######################### DO NOT CHANGE THIS CELL ##########################

def load_dataset(json_path: str) -> Dataset:
    with open(json_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    dataset = []
    for line in lines:
        item = json.loads(line)
        dataset.append(
            {
                "en": item["translation"]["en"],
                "pl": item["translation"]["pl"],
                "keywords": ",".join(item.get("keywords", []))
            }
        )

    return Dataset.from_list(dataset)


train_dataset = load_dataset("train_dataset.jsonl")
print(f"Loaded {len(train_dataset)} training examples.")

val_dataset = load_dataset("valid_dataset.jsonl")
print(f"Loaded {len(val_dataset)} validation examples.")

Loaded 2808 training examples.
Loaded 856 validation examples.


In [7]:
train_dataset[5]

{'en': 'Few-shot inference techniques often utilize meta-learning to generalize better.',
 'pl': 'Few-shot inference techniki często wykorzystują meta-learning, aby uogólnić lepiej.',
 'keywords': 'few-shot inference,meta-learning'}

## Evaluation Code

Code similar to the one below will be used to evaluate the solution on the test set.

In [8]:
######################### DO NOT CHANGE THIS CELL ##########################

def evaluate_model(model, tokenizer, dataset, process_example_fn, batch_size=64, device="cuda", verbose=True):
    """
    Function that evaluates the model using BLEU metric.

    Args:
        model: Model to evaluate.
        tokenizer: Tokenizer used for text processing.
        dataset: Dataset for evaluation.
        process_example_fn: Function that processes examples:
            (en_sentence: str, keywords: List[str]) -> (input_sentence: str).
        verbose: Whether to display details for the first few examples.

    Returns:
        float: Average BLEU score for the dataset.
    """
    model.eval()
    model.to(device)

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)

    bleu_scores = []

    for batch_idx, batch in enumerate(dataloader):
        orig_sentences = batch["en"]
        reference_translations = batch["pl"]
        keywords = batch["keywords"]

        model_inputs = [
            process_example_fn(orig, kws)
            for orig, kws in zip(orig_sentences, keywords)
        ]

        inputs = tokenizer(
            model_inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=64)

        hypotheses = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for i, (ref, hyp) in enumerate(zip(reference_translations, hypotheses)):
            score = sentence_bleu([ref], hyp)
            bleu_scores.append(score)

            if verbose and batch_idx * batch_size + i < 5:
                print(f"Example {batch_idx * batch_size + i}:")
                print(f"Original: {orig_sentences[i]}")
                print(f"Processed: {model_inputs[i]}")
                print(f"Reference: {ref}")
                print(f"Hypothesis: {hyp}")
                print(f"BLEU score: {score:.4f}")
                print("-" * 10)

    bleu_score = sum(bleu_scores) / len(bleu_scores)
    return bleu_score


def compute_score(bleu_score: float) -> float:
    """
    Computes final score based on BLEU metric value.
    """
    lower_bound = 0.82
    upper_bound = 0.86

    if bleu_score <= lower_bound:
        return 0
    elif lower_bound < bleu_score < upper_bound:
        return int(round(100 * (bleu_score - lower_bound) / (upper_bound - lower_bound)))
    else:
        return 100

## Your Solution

In this section, you should implement the `process_example` function, train the model, and save it as a variable named `my_model`.

In [60]:
def process_example(en: str, keywords: str) -> str:
    """
    Funkcja, która przetwarza przykłady ewaluacyjne na tekst
    wejściowy do modelu.
    W czasie treningu możesz, ale nie musisz korzystać z tej funkcji.
    Argumenty:
        en: Tekst w języku angielskim.
        keywords: Słowa kluczowe oddzielone przecinkami.
    """
    # TODO: Zaimplementuj funkcję

    start_token = "<kw>"
    end_token = "</kw>"

    for n, word in enumerate(keywords.split(",")):

      word = word.strip()
      i = en.lower().find(word.lower())

      if i != -1:
        en = en[:i] + start_token + en[i : i + len(word)] + end_token + en[i + len(word):]

    return en

process_example(train_dataset[0]['en'], train_dataset[0]['keywords'])

'Developing <kw>explainable AI</kw> tools is crucial for trust in automated systems.'

In [61]:
len(train_dataset)

2808

In [62]:
# TODO: Model training.
# You are not allowed to change the tokenizer.
# Remember that you have access to the model and tokenizer "gsarti/opus-mt-tc-en-pl".

tokenizer = AutoTokenizer.from_pretrained("gsarti/opus-mt-tc-en-pl")
model = AutoModelForSeq2SeqLM.from_pretrained("gsarti/opus-mt-tc-en-pl").float().to(device)

tokenizer.add_tokens(['<kw>', '</kw>'])

model.resize_token_embeddings(len(tokenizer))

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Embedding(65003, 512, padding_idx=65000)

In [72]:
from torch.utils.data import DataLoader
from tqdm import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True
)


epochs = 16

for epoch in range(epochs):
  model.train()

  losses = []

  for n_batch, X in tqdm(enumerate(train_loader), total=len(train_loader)):
    en, pl, keywords = X['en'], X['pl'], X['keywords']

    en = [process_example(e, k) for e, k in zip(en, keywords)]


    x_in = tokenizer(en, text_target=pl, return_tensors='pt', padding=True, truncation=True)
    x_in = {k: v.to(device) for k, v in x_in.items()}

    outputs = model(**x_in)

    loss = outputs.loss
    #print(loss)
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()


  print(f"Epoch: {epoch}")
  print(f"Loss: {sum(losses) / len(losses):.4f}\n")

  tokenizer = AutoTokenizer.from_pretrained("gsarti/opus-mt-tc-en-pl")
  bleu_score = evaluate_model(
      model=model,
      tokenizer=tokenizer,
      dataset=val_dataset,
      process_example_fn=process_example,
  )
  print(f"Wynik BLEU zbiorze walidacyjnym: {bleu_score:.4f}")

  if epoch == 0:
    max_bleu_score = -1.0


  if max_bleu_score < bleu_score and bleu_score < 0.87:
    max_bleu_score = bleu_score

  else:
    break

  score = compute_score(bleu_score)
  print(f"Punkty na zbiorze walidacyjnym: {score}")

100%|██████████| 44/44 [00:23<00:00,  1.89it/s]


Epoch: 0
Loss: 0.0164

Example 0:
Original: Explainable AI methods are being used to uncover the reasoning behind model predictions.
Processed: <kw>Explainable AI</kw> methods are being used to uncover the reasoning behind <kw>model predictions</kw>.
Reference: Explainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
Hypothesis: Eksplainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
BLEU score: 0.9681
----------
Example 1:
Original: Few-shot generalization presents challenges for traditional machine learning paradigms.
Processed: <kw>Few-shot generalization</kw> presents challenges for traditional machine learning paradigms.
Reference: Few-shot generalization przedstawia wyzwania dla tradycyjnych paradygmatów uczenia się maszynowego.
Hypothesis: Few-shot generalization stwarza wyzwania dla tradycyjnych paradygmatów uczenia maszynowego.
BLEU score: 0.8527
----------
Example 2:
Original: Meta-learning techniques have shown 

100%|██████████| 44/44 [00:24<00:00,  1.82it/s]


Epoch: 1
Loss: 0.0115

Example 0:
Original: Explainable AI methods are being used to uncover the reasoning behind model predictions.
Processed: <kw>Explainable AI</kw> methods are being used to uncover the reasoning behind <kw>model predictions</kw>.
Reference: Explainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
Hypothesis: Eksplainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
BLEU score: 0.9681
----------
Example 1:
Original: Few-shot generalization presents challenges for traditional machine learning paradigms.
Processed: <kw>Few-shot generalization</kw> presents challenges for traditional machine learning paradigms.
Reference: Few-shot generalization przedstawia wyzwania dla tradycyjnych paradygmatów uczenia się maszynowego.
Hypothesis: Few-shot generalization stwarza wyzwania dla tradycyjnych paradygmatów uczenia maszynowego.
BLEU score: 0.8527
----------
Example 2:
Original: Meta-learning techniques have shown 

100%|██████████| 44/44 [00:23<00:00,  1.84it/s]


Epoch: 2
Loss: 0.0097

Example 0:
Original: Explainable AI methods are being used to uncover the reasoning behind model predictions.
Processed: <kw>Explainable AI</kw> methods are being used to uncover the reasoning behind <kw>model predictions</kw>.
Reference: Explainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
Hypothesis: Eksplainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
BLEU score: 0.9681
----------
Example 1:
Original: Few-shot generalization presents challenges for traditional machine learning paradigms.
Processed: <kw>Few-shot generalization</kw> presents challenges for traditional machine learning paradigms.
Reference: Few-shot generalization przedstawia wyzwania dla tradycyjnych paradygmatów uczenia się maszynowego.
Hypothesis: Few-shot generalization stwarza wyzwania dla tradycyjnych paradygmatów uczenia maszynowego.
BLEU score: 0.8527
----------
Example 2:
Original: Meta-learning techniques have shown 

100%|██████████| 44/44 [00:23<00:00,  1.87it/s]


Epoch: 3
Loss: 0.0091

Example 0:
Original: Explainable AI methods are being used to uncover the reasoning behind model predictions.
Processed: <kw>Explainable AI</kw> methods are being used to uncover the reasoning behind <kw>model predictions</kw>.
Reference: Explainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
Hypothesis: Eksplainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
BLEU score: 0.9681
----------
Example 1:
Original: Few-shot generalization presents challenges for traditional machine learning paradigms.
Processed: <kw>Few-shot generalization</kw> presents challenges for traditional machine learning paradigms.
Reference: Few-shot generalization przedstawia wyzwania dla tradycyjnych paradygmatów uczenia się maszynowego.
Hypothesis: Few-shot generalization stwarza wyzwania dla tradycyjnych paradygmatów uczenia maszynowego.
BLEU score: 0.8527
----------
Example 2:
Original: Meta-learning techniques have shown 

In [73]:
my_model = model

## Evaluation

Running the cell below will allow you to check how many points your solution would achieve on the validation data.

Before submission, make sure that the entire notebook runs from start to finish without errors after setting the flag *FINAL_EVALUATION_MODE = True*, and without requiring any user interaction after selecting “Run All”.

In [74]:
######################### DO NOT CHANGE THIS CELL ##########################
FINAL_EVALUATION_MODE = True

if FINAL_EVALUATION_MODE==True:
    tokenizer = AutoTokenizer.from_pretrained("gsarti/opus-mt-tc-en-pl")
    bleu_score = evaluate_model(
        model=my_model,
        tokenizer=tokenizer,
        dataset=val_dataset,
        process_example_fn=process_example,
    )
    print(f"BLEU score on validation set: {bleu_score:.4f}")

    score = compute_score(bleu_score)
    print(f"Points on validation set: {score}")

Example 0:
Original: Explainable AI methods are being used to uncover the reasoning behind model predictions.
Processed: <kw>Explainable AI</kw> methods are being used to uncover the reasoning behind <kw>model predictions</kw>.
Reference: Explainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
Hypothesis: Eksplainable AI metody są wykorzystywane do odkrywania rozumowania za model predictions.
BLEU score: 0.9681
----------
Example 1:
Original: Few-shot generalization presents challenges for traditional machine learning paradigms.
Processed: <kw>Few-shot generalization</kw> presents challenges for traditional machine learning paradigms.
Reference: Few-shot generalization przedstawia wyzwania dla tradycyjnych paradygmatów uczenia się maszynowego.
Hypothesis: Few-shot generalization stwarza wyzwania dla tradycyjnych paradygmatów uczenia maszynowego.
BLEU score: 0.8527
----------
Example 2:
Original: Meta-learning techniques have shown promise in adapting mod

During evaluation, the model will be saved to a file `your_model.pkl`, and the processing function to a file `your_function.pkl`, and they will be evaluated on the test set.

In [16]:
######################### DO NOT CHANGE THIS CELL ##########################

if FINAL_EVALUATION_MODE:
    import cloudpickle

    OUTPUT_PATH = "file_output"

    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "your_function.pkl")
    MODEL_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "your_model.pkl")

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(process_example, f)

    with open(MODEL_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(my_model, f)